In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining"

!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/Othercomputers/My PC/Neural_Chess/NeuralChess.Engine/ModelTraining


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from model import ChessValueNet
from dataset_manager import ChunkedChessDataset
from datetime import datetime

GPU = "cuda"
EPOCHS = 20

device = torch.device(GPU if torch.cuda.is_available() else "cpu")
model = ChessValueNet().to(device)

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.002)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

scaler = GradScaler(GPU)

dataset = ChunkedChessDataset(folder_path="./train_dataset_chunks")
train_loader = DataLoader(dataset, batch_size=4096)

model.train()
print(f"Starting training loop on device: {device}")

for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    batch_count = 0

    for batch_boards, batch_scores in train_loader:
        batch_boards = batch_boards.to(device)
        batch_scores = batch_scores.to(device).float().view(-1, 1)

        optimizer.zero_grad()

        with autocast(device_type=GPU, dtype=torch.float16):
            predictions = model(batch_boards)
            loss = criterion(predictions, batch_scores)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        batch_count += 1

        if batch_count % 500 == 0:
            print(f"Epoch {epoch} | Batch {batch_count} | Current Loss: {loss.item():.4f} | Time: {datetime.now()}")

    scheduler.step()
    print(f"=== Epoch {epoch} Complete! Average Loss: {running_loss / batch_count:.4f} ===")

torch.save(model.state_dict(), "chess_model_weights.pth")
print("Weights successfully saved to disk!")

Starting training loop on device: cuda
Epoch 1 | Batch 500 | Current Loss: 8.6800 | Time: 2026-06-14 22:28:16.122138
Epoch 1 | Batch 1000 | Current Loss: 6.5435 | Time: 2026-06-14 22:28:39.630269
Epoch 1 | Batch 1500 | Current Loss: 6.2930 | Time: 2026-06-14 22:29:03.927358
Epoch 1 | Batch 2000 | Current Loss: 5.7309 | Time: 2026-06-14 22:29:27.734664
Epoch 1 | Batch 2500 | Current Loss: 5.4414 | Time: 2026-06-14 22:29:51.632795
Epoch 1 | Batch 3000 | Current Loss: 5.1374 | Time: 2026-06-14 22:30:15.491717
Epoch 1 | Batch 3500 | Current Loss: 4.7805 | Time: 2026-06-14 22:30:39.032695
Epoch 1 | Batch 4000 | Current Loss: 5.0074 | Time: 2026-06-14 22:31:02.583900
Epoch 1 | Batch 4500 | Current Loss: 5.3975 | Time: 2026-06-14 22:31:26.653208
Epoch 1 | Batch 5000 | Current Loss: 4.6864 | Time: 2026-06-14 22:31:50.316995
Epoch 1 | Batch 5500 | Current Loss: 4.5822 | Time: 2026-06-14 22:32:13.668929
Epoch 1 | Batch 6000 | Current Loss: 5.4183 | Time: 2026-06-14 22:32:36.865289
Epoch 1 | Batc

KeyboardInterrupt: 